# Trace-Based Agent Evaluation without an LLM Judge

## Overview

A final answer can look plausible even when an agent called the wrong tool, sent the wrong arguments, ignored its evidence, or exceeded a latency budget. This tutorial evaluates the *execution trace* instead of asking another language model for a subjective score. It produces reproducible per-case diagnostics and a suite-level quality gate suitable for local regression tests or CI.

Everything runs offline with the Python standard library. The two agents are deterministic fixtures: one contains realistic routing mistakes, while the improved version demonstrates the same cases after the defects are corrected.

## Detailed Explanation

### What a trace tells us

A useful trace records ordered tool calls, structured evidence, structured claims, a canonical rendered answer, latency, and any execution error. A test case declares the complete tool contract and evidence. The adapter renders the final answer from claims, removing independently generated prose as a second source of truth.

### Agent Architecture

![Trace-Based Agent Evaluation](../images/trace-based-agent-evaluation.svg)

The evaluator is deliberately outside the agent. It consumes a trace through a small data contract, so the same scorer can compare a LangGraph workflow, a custom loop, or a hosted agent as long as each adapter emits the required fields.

## Required Packages

### No installation required

The implementation uses `dataclasses`, `statistics`, and other Python standard-library modules.

In [ ]:
from dataclasses import dataclass
from math import ceil
from typing import Any, Callable, Dict, List, Optional

## Implementation

### Define evaluation cases and trace contracts

Each case specifies only behavior that can be checked deterministically. `expected_calls` describes the complete allowed call sequence, while `expected_evidence` is compared with structured tool evidence and structured claims. The adapter derives a canonical answer from those claims so contradictory prose cannot receive credit.

In [ ]:
@dataclass(frozen=True)
class EvalCase:
    """Frozen tool-sequence, evidence, and latency expectations for one prompt."""
    case_id: str
    prompt: str
    expected_calls: List[Dict[str, Any]]
    expected_evidence: Dict[str, Any]
    max_latency_ms: int


@dataclass(frozen=True)
class AgentTrace:
    """Framework-neutral record of one agent execution."""
    answer: str
    tool_calls: List[Dict[str, Any]]
    evidence: Dict[str, Any]
    claims: Dict[str, Any]
    latency_ms: int
    error: Optional[str] = None


@dataclass(frozen=True)
class TraceScore:
    """Deterministic check results and weighted total for one case."""
    case_id: str
    total: float
    checks: Dict[str, bool]
    failed_checks: List[str]

### Build a small regression dataset

The suite covers weather and order lookup with different argument shapes. Expected values are literal fixtures, which keeps the evaluator independent from agent implementation.

In [ ]:
EVAL_CASES = [
    EvalCase(
        case_id="weather-paris",
        prompt="What is the weather in Paris?",
        expected_calls=[{"name": "get_weather", "args": {"city": "Paris"}}],
        expected_evidence={"temperature": "18 C"},
        max_latency_ms=500,
    ),
    EvalCase(
        case_id="weather-tokyo",
        prompt="Do I need an umbrella in Tokyo?",
        expected_calls=[{"name": "get_weather", "args": {"city": "Tokyo"}}],
        expected_evidence={"condition": "rain"},
        max_latency_ms=500,
    ),
    EvalCase(
        case_id="order-a100",
        prompt="Has order A-100 shipped?",
        expected_calls=[{"name": "lookup_order", "args": {"order_id": "A-100"}}],
        expected_evidence={"status": "shipped"},
        max_latency_ms=500,
    ),
    EvalCase(
        case_id="order-b200",
        prompt="Where is order B-200?",
        expected_calls=[{"name": "lookup_order", "args": {"order_id": "B-200"}}],
        expected_evidence={"status": "processing"},
        max_latency_ms=500,
    ),
]

### Score one execution trace

The four checks have explicit weights: correct tool sequence 25%, correct argument sequence 25%, grounded evidence 25%, and latency 25%. An execution error receives zero because a fast failure with no result should not earn partial production credit. Evidence must exactly match the structured tool result and claims, and the answer must be their deterministic rendering.

In [ ]:
WEIGHTS = {"tool": 0.25, "arguments": 0.25, "evidence": 0.25, "latency": 0.25}


def render_claims(claims: Dict[str, Any]) -> str:
    """Render structured claims into the only accepted answer representation."""
    return "; ".join(f"{key}: {claims[key]}" for key in sorted(claims))


def score_trace(case: EvalCase, trace: AgentTrace) -> TraceScore:
    """Score a complete trace against one explicit behavioral contract."""
    if trace.error:
        checks = {name: False for name in WEIGHTS}
        return TraceScore(case.case_id, 0.0, checks, ["error", *checks.keys()])

    actual_tools = [call.get("name") for call in trace.tool_calls]
    expected_tools = [call["name"] for call in case.expected_calls]
    actual_args = [call.get("args") for call in trace.tool_calls]
    expected_args = [call["args"] for call in case.expected_calls]
    checks = {
        "tool": actual_tools == expected_tools,
        "arguments": actual_args == expected_args,
        "evidence": (
            trace.evidence == case.expected_evidence
            and trace.claims == case.expected_evidence
            and trace.answer == render_claims(trace.claims)
        ),
        "latency": trace.latency_ms <= case.max_latency_ms,
    }
    total = round(sum(WEIGHTS[name] for name, passed in checks.items() if passed), 2)
    return TraceScore(
        case_id=case.case_id,
        total=total,
        checks=checks,
        failed_checks=[name for name, passed in checks.items() if not passed],
    )

### Create baseline and improved agent fixtures

Both functions emit the same production-facing trace contract. The baseline routes one order question to the wrong tool and sends the wrong city for one weather question. The improved agent repairs both defects. Latencies are recorded constants so the tutorial is reproducible.

In [ ]:
TRACE_FIXTURES = {
    "weather-paris": ("get_weather", {"city": "Paris"}, {"temperature": "18 C"}, 120),
    "weather-tokyo": ("get_weather", {"city": "Tokyo"}, {"condition": "rain"}, 180),
    "order-a100": ("lookup_order", {"order_id": "A-100"}, {"status": "shipped"}, 90),
    "order-b200": ("lookup_order", {"order_id": "B-200"}, {"status": "processing"}, 110),
}


def trace_from_fixture(case: EvalCase, tool: str, args: Dict[str, Any]) -> AgentTrace:
    """Emit a deterministic trace while allowing routing defects to be injected."""
    _, _, evidence, latency = TRACE_FIXTURES[case.case_id]
    return AgentTrace(
        answer=render_claims(evidence),
        tool_calls=[{"name": tool, "args": args}],
        evidence=evidence,
        claims=evidence.copy(),
        latency_ms=latency,
    )


def baseline_agent(case: EvalCase) -> AgentTrace:
    """Return traces with two deliberate routing regressions."""
    expected_tool, expected_args, _, _ = TRACE_FIXTURES[case.case_id]
    if case.case_id == "weather-tokyo":
        return trace_from_fixture(case, expected_tool, {"city": "Kyoto"})
    if case.case_id == "order-b200":
        return trace_from_fixture(case, "search_web", expected_args)
    return trace_from_fixture(case, expected_tool, expected_args)


def improved_agent(case: EvalCase) -> AgentTrace:
    """Return the corrected deterministic trace for every case."""
    expected_tool, expected_args, _, _ = TRACE_FIXTURES[case.case_id]
    return trace_from_fixture(case, expected_tool, expected_args)

### Aggregate the suite

A case passes only at full credit. The report keeps per-case scores for diagnosis and summarizes pass rate, tool accuracy, and nearest-rank p95 latency.

In [ ]:
Agent = Callable[[EvalCase], AgentTrace]


def percentile(values: List[int], percentile_value: float) -> int:
    """Return a nearest-rank percentile for a non-empty latency list."""
    ordered = sorted(values)
    index = max(0, ceil(percentile_value * len(ordered)) - 1)
    return ordered[index]


def evaluate_suite(agent: Agent, cases: List[EvalCase]) -> Dict[str, Any]:
    """Run frozen cases and aggregate diagnostics and suite metrics."""
    runs = []
    for case in cases:
        trace = agent(case)
        runs.append({"case": case, "trace": trace, "score": score_trace(case, trace)})

    return {
        "case_count": len(runs),
        "pass_rate": sum(run["score"].total == 1.0 for run in runs) / len(runs),
        "tool_accuracy": sum(run["score"].checks["tool"] for run in runs) / len(runs),
        "p95_latency_ms": percentile([run["trace"].latency_ms for run in runs], 0.95),
        "runs": runs,
    }

### Turn metrics into a quality gate

The gate returns structured failures instead of raising immediately, so CI can print all regressions in one run. Higher is better for pass rate and tool accuracy; lower is better for latency.

In [ ]:
def check_quality_gate(
    report: Dict[str, Any], thresholds: Dict[str, float]
) -> Dict[str, Any]:
    """Return every metric regression against inclusive thresholds."""
    failures = []
    if report["pass_rate"] < thresholds["pass_rate"]:
        failures.append(
            f"pass_rate {report['pass_rate']:.2f} < {thresholds['pass_rate']:.2f}"
        )
    if report["tool_accuracy"] < thresholds["tool_accuracy"]:
        failures.append(
            f"tool_accuracy {report['tool_accuracy']:.2f} < {thresholds['tool_accuracy']:.2f}"
        )
    if report["p95_latency_ms"] > thresholds["p95_latency_ms"]:
        failures.append(
            f"p95_latency_ms {report['p95_latency_ms']} > {thresholds['p95_latency_ms']}"
        )
    return {"passed": not failures, "failures": failures}

## Usage Example

### Compare baseline and improved agents

The baseline passes two of four cases. The case-level output shows whether the failure came from routing, arguments, evidence, latency, or execution itself.

In [ ]:
baseline_report = evaluate_suite(baseline_agent, EVAL_CASES)
improved_report = evaluate_suite(improved_agent, EVAL_CASES)

for label, report in [("baseline", baseline_report), ("improved", improved_report)]:
    print(
        label,
        {
            "pass_rate": report["pass_rate"],
            "tool_accuracy": report["tool_accuracy"],
            "p95_latency_ms": report["p95_latency_ms"],
        },
    )
    for run in report["runs"]:
        print(" ", run["score"].case_id, run["score"].failed_checks)

### Enforce the regression threshold

A repository test can assert on the final `passed` field or exit non-zero when it is false.

In [ ]:
THRESHOLDS = {"pass_rate": 1.0, "tool_accuracy": 1.0, "p95_latency_ms": 500}
baseline_gate = check_quality_gate(baseline_report, THRESHOLDS)
improved_gate = check_quality_gate(improved_report, THRESHOLDS)

assert baseline_gate == {
    "passed": False,
    "failures": ["pass_rate 0.50 < 1.00", "tool_accuracy 0.75 < 1.00"],
}
assert improved_gate == {"passed": True, "failures": []}
print("Improved agent passes the quality gate")

## Comparison

| Evaluation method | Reproducible | Diagnoses tool behavior | Handles open-ended prose | Cost |
|---|---:|---:|---:|---:|
| Exact answer matching | High | No | Poorly | Low |
| LLM-as-judge | Depends on model and prompt | Sometimes | Well | Model calls |
| Trace-based checks (this tutorial) | High | Yes | Through evidence contracts | Low |
| Human review | With a rubric | Yes | Best | High |

Trace checks are strongest for contracts a machine can observe: tool routing, parameters, structured claims, errors, ordering, latency, and cost. The canonical answer in this tutorial deliberately does not evaluate tone or nuanced synthesis. Mature evaluations combine deterministic gates for non-negotiable behavior with sampled human or calibrated judge review for subjective natural-language quality.

## Additional Considerations

- **Freeze the dataset.** Version cases and thresholds together so metric changes have an attributable cause. Keep a separate hidden set if prompts might overfit public cases.
- **Redact traces.** Arguments and tool results can contain personal or secret data. Store the minimum fields needed for evaluation and apply retention limits.
- **Measure autonomous success separately.** A run completed after human escalation is useful operationally but is not an autonomous pass; record both outcomes.
- **Use distributions for real latency.** Four deterministic fixtures make the mechanics easy to see. Production gates need repeated runs, warm/cold separation, timeouts, and enough samples for percentiles.
- **Do not reward fast errors.** This scorer zeros every check when the trace has an error. Report error rate independently as suites grow.
- **Adapt at the boundary.** Convert framework-specific spans into `AgentTrace` rather than embedding LangGraph, LangSmith, OpenTelemetry, or vendor assumptions in the scorer.
- **Audit the evaluator.** Hand-check literals and write mutation tests for wrong tools, arguments, missing evidence, slow traces, and errors. A broken gate can create more confidence than no gate.

## References

- [LangSmith evaluation concepts](https://docs.smith.langchain.com/evaluation)
- [OpenTelemetry trace specification](https://opentelemetry.io/docs/concepts/signals/traces/)
- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)